# Olist — Chapter 11: Statistical ValidationTurns the descriptive findings into **quantified, controlled, significance-tested claims**.Three blocks:1. **OLS** — what actually drives review score, controlling for everything else2. **Logistic regression** — does a bad first order reduce the odds of returning?3. **Significance tests** — ANOVA, chi-square, and confidence intervals on the group gaps> Run the Setup cells from the figures notebook first (auth + `q()` helper), or use the> self-contained setup below.

## 1 · Setup

In [ ]:
!pip install statsmodels --quietfrom google.colab import authauth.authenticate_user()import warnings; warnings.filterwarnings("ignore")import pandas as pd, numpy as npimport statsmodels.api as smimport statsmodels.formula.api as smffrom scipy import statsfrom google.cloud import bigqueryPROJECT = "blinkit-analysis-503505"DATASET = "olist"client  = bigquery.Client(project=PROJECT)def q(sql):    return client.query(sql.replace("`olist.", f"`{PROJECT}.{DATASET}.")).to_dataframe()print("ready")

## 2 · Build the modelling tableOne row per **order** (not per item), so each observation is independent.`v_fact_order_items` is item-level, so we aggregate up.

In [ ]:
raw = q("""WITH item_agg AS (  SELECT    order_id,    SUM(price)              AS price,    SUM(freight_value)      AS freight,    COUNT(*)                AS n_items,    AVG(weight_g)           AS weight_g,    AVG(photos_qty)         AS photos_qty,    AVG(distance_km)        AS distance_km,    ANY_VALUE(category)     AS category,    ANY_VALUE(seller_state) AS seller_state,    MAX(CAST(is_interstate AS INT64)) AS is_interstate  FROM `olist.v_fact_order_items`  WHERE order_status = 'delivered'  GROUP BY order_id),cust_orders AS (  SELECT customer_unique_id, COUNT(DISTINCT order_id) AS n_orders  FROM `olist.v_orders_enriched` GROUP BY customer_unique_id),seq AS (  SELECT customer_unique_id, order_id,         ROW_NUMBER() OVER (PARTITION BY customer_unique_id ORDER BY purchase_ts) AS order_seq  FROM `olist.v_orders_enriched`)SELECT  o.order_id, o.customer_unique_id, o.customer_state,  DATE_DIFF(DATE(o.delivered_ts), DATE(o.purchase_ts),  DAY) AS delivery_days,  DATE_DIFF(DATE(o.delivered_ts), DATE(o.estimated_ts), DAY) AS lateness_days,  r.review_score,  i.price, i.freight, i.n_items, i.weight_g, i.photos_qty,  i.distance_km, i.category, i.is_interstate,  s.order_seq,  c.n_orders,  IF(c.n_orders > 1, 1, 0) AS repeat_customerFROM `olist.v_orders_enriched` oJOIN item_agg i USING (order_id)JOIN seq s ON o.order_id = s.order_idJOIN cust_orders c ON o.customer_unique_id = c.customer_unique_idLEFT JOIN `olist.v_reviews_clean` r ON o.order_id = r.order_idWHERE o.order_status = 'delivered' AND o.delivered_ts IS NOT NULL""")print("orders:", len(raw))raw.head()

In [ ]:
# Feature engineeringdf = raw.copy()# Keep the 12 largest categories, bucket the rest -> avoids 70 dummy variablestop_cats = df.category.value_counts().head(12).indexdf["cat"] = np.where(df.category.isin(top_cats), df.category, "other")# Logs for the heavily right-skewed money/distance variablesdf["log_price"] = np.log1p(df.price)df["log_dist"]  = np.log1p(df.distance_km)# Lateness split into two interpretable piecesdf["late_flag"]     = (df.lateness_days > 0).astype(int)   # was it late at alldf["days_late_pos"] = df.lateness_days.clip(lower=0)       # how late, 0 if earlyprint(df[["delivery_days","lateness_days","review_score","distance_km","price"]].describe().round(2))

## 3 · OLS — what drives review score?Replaces *"late orders get worse reviews"* with a controlled, quantified statement.**Note on specification:** `delivery_days` and `days_late_pos` are correlated, so we fitthree nested models to see how much each adds. Report the one you can defend.

In [ ]:
model_df = df[["review_score","days_late_pos","late_flag","delivery_days","log_dist",               "log_price","freight","n_items","is_interstate","cat"]].dropna()m1 = smf.ols("review_score ~ days_late_pos", data=model_df).fit()m2 = smf.ols("review_score ~ days_late_pos + delivery_days", data=model_df).fit()m3 = smf.ols("review_score ~ days_late_pos + delivery_days + log_dist + log_price "             "+ freight + n_items + is_interstate + C(cat)", data=model_df).fit()print(f"{'Model':<34}{'N':>8}{'R2':>9}{'Adj R2':>9}")for name, m in [("1. lateness only", m1),                ("2. + delivery time", m2),                ("3. + controls & category", m3)]:    print(f"{name:<34}{int(m.nobs):>8}{m.rsquared:>9.4f}{m.rsquared_adj:>9.4f}")print("\n--- Full model coefficients ---")for v in ["days_late_pos","delivery_days","log_dist","log_price",          "freight","n_items","is_interstate"]:    b, p = m3.params[v], m3.pvalues[v]    star = "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "n.s."    print(f"  {v:<16} beta = {b:+.5f}   p = {p:.3g}  {star}")

In [ ]:
# Full statsmodels table — put this in the report appendixprint(m3.summary().tables[1])

## 4 · Logistic regression — does a bad first order kill retention?**This is the key test of the causal chain.** Restricted to each customer's FIRST order,so we are asking: *given how the first purchase went, did they ever come back?*

In [ ]:
first = df[df.order_seq == 1].copy()fit_df = first[["repeat_customer","review_score","days_late_pos","delivery_days",                "log_dist","log_price","n_items"]].dropna()logit = smf.logit("repeat_customer ~ review_score + days_late_pos + delivery_days "                  "+ log_dist + log_price + n_items", data=fit_df).fit(disp=0)print(f"N = {int(logit.nobs):,}   |   base repeat rate = {100*fit_df.repeat_customer.mean():.2f}%")print(f"Pseudo R2 = {logit.prsquared:.4f}\n")print(f"{'variable':<18}{'odds ratio':>12}{'95% CI':>22}{'p':>12}")ci = np.exp(logit.conf_int())for v in ["review_score","days_late_pos","delivery_days","log_dist","log_price","n_items"]:    orr = np.exp(logit.params[v]); lo, hi = ci.loc[v]    p = logit.pvalues[v]    star = "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "n.s."    print(f"{v:<18}{orr:>12.4f}{f'[{lo:.3f}, {hi:.3f}]':>22}{p:>12.3g}  {star}")print("\nOR > 1 raises the odds of returning; OR < 1 lowers them.")

In [ ]:
# Unadjusted view: return rate by first-order review score, with 95% CIsg = first.dropna(subset=["review_score"]).groupby("review_score").repeat_customer.agg(["mean","count"])g["se"]   = np.sqrt(g["mean"] * (1 - g["mean"]) / g["count"])g["lo"]   = 100 * (g["mean"] - 1.96 * g["se"])g["hi"]   = 100 * (g["mean"] + 1.96 * g["se"])g["rate"] = 100 * g["mean"]print(g[["count","rate","lo","hi"]].round(2))print("\nIf these confidence intervals overlap, first-order satisfaction does NOT "      "meaningfully predict return behaviour.")

## 5 · Significance tests on the group differencesAnswers *"is that gap real, or noise?"* for the headline comparisons.

In [ ]:
# ANOVA — review score across lateness bucketsmdl = model_df.copy()mdl["bucket"] = pd.cut(mdl.days_late_pos, [-1, 0, 3, 7, 10**6],                       labels=["On time / early", "1-3d late", "4-7d late", "8d+ late"])groups = [mdl.review_score[mdl.bucket == b].values for b in mdl.bucket.cat.categories]F, p = stats.f_oneway(*groups)print(f"ANOVA  review_score ~ lateness bucket:  F = {F:,.1f},  p = {p:.3g}\n")print(mdl.groupby("bucket").review_score.agg(["count","mean","std"]).round(3))

In [ ]:
# Chi-square — late delivery vs a 1-2 star ratingmdl["bad_review"] = (mdl.review_score <= 2).astype(int)ct = pd.crosstab(mdl.late_flag, mdl.bad_review)chi2, p, dof, _ = stats.chi2_contingency(ct)late_rate = 100 * ct.loc[1,1] / ct.loc[1].sum()ontime_rate = 100 * ct.loc[0,1] / ct.loc[0].sum()print(f"Chi-square: chi2 = {chi2:,.1f}, dof = {dof}, p = {p:.3g}")print(f"  1-2 star rate when LATE    : {late_rate:.1f}%")print(f"  1-2 star rate when ON TIME : {ontime_rate:.1f}%")print(f"  Relative risk              : {late_rate/ontime_rate:.1f}x")# Cramer's V — effect size, not just significancen = ct.values.sum()print(f"  Cramer's V (effect size)   : {np.sqrt(chi2/(n*(min(ct.shape)-1))):.3f}")

In [ ]:
# Confidence intervals on repeat rate by distance band (the causal-chain table)ch = df[df.order_seq == 1].dropna(subset=["distance_km"]).copy()ch["band"] = pd.cut(ch.distance_km, [-1, 200, 500, 1000, 10**6],                    labels=["0-200 km","200-500 km","500-1000 km","1000+ km"])out = ch.groupby("band").repeat_customer.agg(["mean","count"])out["se"]   = np.sqrt(out["mean"] * (1 - out["mean"]) / out["count"])out["rate"] = 100 * out["mean"]out["lo"]   = 100 * (out["mean"] - 1.96 * out["se"])out["hi"]   = 100 * (out["mean"] + 1.96 * out["se"])print(out[["count","rate","lo","hi"]].round(2))print("\nOverlapping CIs across bands mean distance does NOT drive retention directly.")

## 6 · Reporting templateFill these in from the output above:- *"Controlling for category, price, distance and basket size, each additional day late reduces  review score by **β** points (p **&lt;** .001); total delivery time carries the larger effect at  **β** points per day. The model explains **R²** of variance in review score."*- *"Late deliveries are rated 1–2 star **X×** more often than on-time ones  (χ² = **…**, p **&lt;** .001, Cramér's V = **…**)."*- *"First-order review score **[did / did not]** significantly predict repeat purchase  (OR = **…**, 95% CI **…**, p = **…**)."***Report the null results too.** A finding that satisfaction does *not* drive retention is areal finding — and defending it is a stronger interview signal than reporting only whatconfirms your hypothesis.